# TSForecasting: Preprocessing and Advanced Components

This notebook demonstrates the internal preprocessing mechanisms and decomposed components of TSForecasting for advanced usage.

## Topics Covered

- `Processing` class and time series transformation
- Lag feature generation (window creation)
- Horizon target generation (multi-step)
- DateTime feature engineering
- Expanding window evaluation logic
- Direct model usage with `BaseForecaster`
- `FORECASTER_CLASSES` dictionary
- Metric strategies and evaluation

In [ ]:
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=Warning)

from tsforecasting import Processing, TimeSeriesDatasetGenerator
from tsforecasting.evaluation.metrics import METRIC_STRATEGIES
from tsforecasting.models import (
    FORECASTER_CLASSES,
    RandomForestForecaster,
    XGBoostForecaster,
)

---
## 1. Time Series Transformation

The `Processing` class transforms raw time series data into a supervised learning format with lag features and horizon targets.

```
Raw data:       Date, y
                  |
                  v
Transformed:    Date, y_lag_1, y_lag_2, ..., y_horizon_1, y_horizon_2, ...
```

### 1.1 Generate Raw Data

In [ ]:
data = TimeSeriesDatasetGenerator.generate(
    n_samples=100,
    granularity="1mo",
    patterns=["trend", "seasonal"],
    random_state=42,
)

print(f"Raw data shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
data.head()

### 1.2 Processing Class Configuration

In [ ]:
processor = Processing()

# Configuration
LAGS = 10
HORIZON = 5
GRANULARITY = "1mo"

print("Configuration:")
print(f"  LAGS (window_size): {LAGS}")
print(f"  HORIZON: {HORIZON}")
print(f"  GRANULARITY: {GRANULARITY}")

### 1.3 make_timeseries() Transformation

In [ ]:
timeseries = processor.make_timeseries(
    dataset=data,
    window_size=LAGS,
    horizon=HORIZON,
    datetime_engineering=True,
)

print(f"Transformed shape: {timeseries.shape}")
print(f"Rows: {len(data)} -> {len(timeseries)} (reduced due to windowing)")

### 1.4 Understanding Generated Columns

In [ ]:
columns = timeseries.columns.tolist()

# Categorize columns
lag_cols = [c for c in columns if c.startswith("y_lag_")]
horizon_cols = [c for c in columns if c.startswith("y_horizon_")]
date_cols = [c for c in columns if c.startswith("Date_")]
other_cols = [
    c
    for c in columns
    if c not in lag_cols + horizon_cols + date_cols and c != "Date"
]

print("COLUMN STRUCTURE:")
print(f"\nLAG FEATURES ({len(lag_cols)} columns):")
print(f"  {lag_cols}")
print(f"  -> Previous values: y_lag_1 = t-1, y_lag_2 = t-2, ..., y_lag_{LAGS} = t-{LAGS}")

print(f"\nHORIZON TARGETS ({len(horizon_cols)} columns):")
print(f"  {horizon_cols}")
print(f"  -> Future values: y_horizon_1 = t+1, ..., y_horizon_{HORIZON} = t+{HORIZON}")

print(f"\nDATETIME FEATURES ({len(date_cols)} columns):")
print(f"  {date_cols[:5]}...")

print(f"\nOTHER ({len(other_cols)} columns):")
print(f"  {other_cols}")

In [ ]:
# Visual representation
print("Sample transformation:")
sample_cols = [
    "y_lag_3",
    "y_lag_2",
    "y_lag_1",
    "y_horizon_1",
    "y_horizon_2",
    "y_horizon_3",
]
available = [c for c in sample_cols if c in timeseries.columns]
timeseries[available].head()

---
## 2. Feature and Target Separation

### 2.1 Define Input/Output Columns

In [ ]:
# Input features (X): everything except horizons and Date
input_cols = [
    c for c in timeseries.columns if not c.startswith("y_horizon_") and c != "Date"
]

# Target variables (y): horizon columns
target_cols = [c for c in timeseries.columns if c.startswith("y_horizon_")]

print(f"Input features: {len(input_cols)} columns")
print(f"  Includes: {len(lag_cols)} lags + {len(date_cols)} datetime features")

print(f"\nTarget variables: {len(target_cols)} columns")
print(f"  {target_cols}")

### 2.2 Handle NaN Values

In [ ]:
# The last row(s) have NaN in horizons (future forecasting row)
nan_count = timeseries[target_cols].isna().any(axis=1).sum()
print(f"Rows with NaN in targets: {nan_count}")

# Filter valid rows for training
valid_mask = ~timeseries[target_cols].isna().any(axis=1)
timeseries_clean = timeseries[valid_mask].copy()

print(f"Clean data: {len(timeseries_clean)} rows")

### 2.3 Create NumPy Arrays

In [ ]:
X = timeseries_clean[input_cols].values
y = timeseries_clean[target_cols].values

print(f"X shape: {X.shape} (samples x features)")
print(f"y shape: {y.shape} (samples x horizons)")

---
## 3. Expanding Window Evaluation

Expanding Window evaluation progressively increases the training set:

```
Iteration 1: Train [0:80%], Test [80%:100%]
Iteration 2: Train [0:80%+slide], Test [80%+slide:100%]
...
```

This simulates realistic forecasting where you train on all available historical data and evaluate on future unseen data.

### 3.1 Configure Window Parameters

In [ ]:
TRAIN_SIZE = 0.80
SLIDING_SIZE = 10

total_samples = len(timeseries_clean)
initial_train = int(total_samples * TRAIN_SIZE)
test_size = total_samples - initial_train
iterations = test_size // SLIDING_SIZE

print("Window Configuration:")
print(f"  Total samples: {total_samples}")
print(f"  Initial train: {initial_train} ({TRAIN_SIZE * 100:.0f}%)")
print(f"  Test pool: {test_size}")
print(f"  Sliding size: {SLIDING_SIZE}")
print(f"  Iterations: {iterations}")

### 3.2 Window Evolution

In [ ]:
print("Train/Test splits per iteration:")
for i in range(iterations):
    train_end = initial_train + (i * SLIDING_SIZE)
    test_start = train_end
    test_end = total_samples
    print(
        f"  Iter {i + 1}: Train [0:{train_end}], "
        f"Test [{test_start}:{test_end}] ({test_end - test_start} samples)"
    )

### 3.3 Manual Expanding Window Implementation

In [ ]:
def expanding_window_evaluate(
    X: np.ndarray,
    y: np.ndarray,
    model_class,
    model_params: dict,
    train_size: float,
    sliding_size: int,
) -> pd.DataFrame:
    """Manual expanding window evaluation."""
    total = len(X)
    initial = int(total * train_size)
    iterations = (total - initial) // sliding_size

    results = []

    for i in range(iterations):
        current_train = initial + (i * sliding_size)

        # Split
        X_train, X_test = X[:current_train], X[current_train:]
        y_train, y_test = y[:current_train], y[current_train:]

        # Scale
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # Train model
        model = model_class(**model_params)
        model.fit(X_train_scaled, y_train)

        # Predict
        y_pred = model.predict(X_test_scaled)

        # Evaluate
        mae = mean_absolute_error(y_test, y_pred)

        results.append(
            {
                "window": i + 1,
                "train_size": current_train,
                "test_size": len(X_test),
                "mae": mae,
            }
        )

    return pd.DataFrame(results)

In [ ]:
# Run evaluation
print("Running expanding window evaluation...")
results = expanding_window_evaluate(
    X,
    y,
    model_class=RandomForestForecaster,
    model_params={"n_estimators": 50, "random_state": 42},
    train_size=TRAIN_SIZE,
    sliding_size=SLIDING_SIZE,
)

print(results.to_string(index=False))
print(f"\nMean MAE: {results['mae'].mean():.4f}")
print(f"Std MAE: {results['mae'].std():.4f}")

---
## 4. Direct Model Usage

Models can be used directly without the pipeline for custom workflows. All models inherit from `BaseForecaster` providing a unified interface.

**BaseForecaster Interface:**
- `fit(X, y) -> Self`: Train model
- `predict(X) -> np.ndarray`: Generate predictions
- `get_params() -> Dict`: Get hyperparameters
- `set_params(**params)`: Update parameters
- `is_fitted -> bool`: Check fitted state

### 4.1 Direct Instantiation

In [ ]:
# Prepare data
train_idx = int(len(X) * 0.8)
X_train, X_test = X[:train_idx], X[train_idx:]
y_train, y_test = y[:train_idx], y[train_idx:]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# RandomForestForecaster
rf = RandomForestForecaster(n_estimators=50, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)

print("RandomForestForecaster:")
print(f"  Params: {rf.get_params()}")
print(f"  is_fitted: {rf.is_fitted}")
print(f"  Predictions shape: {rf_pred.shape}")

In [ ]:
# XGBoostForecaster
xgb = XGBoostForecaster(n_estimators=50)
xgb.fit(X_train_scaled, y_train)
xgb_pred = xgb.predict(X_test_scaled)

print("XGBoostForecaster:")
print(f"  Predictions shape: {xgb_pred.shape}")

### 4.2 FORECASTER_CLASSES Dictionary

In [ ]:
print(f"Available classes: {list(FORECASTER_CLASSES.keys())}")

In [ ]:
# Dynamic model selection
print("\nDynamic model selection:")
for name in ["RandomForest", "GBR", "XGBoost"]:
    model_class = FORECASTER_CLASSES[name]
    model = model_class(n_estimators=30 if name != "GBR" else 30)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, pred)
    print(f"  {name}: MAE = {mae:.4f}")

---
## 5. Metrics and Evaluation

### 5.1 Metric Strategies

In [ ]:
print(f"Available metrics: {list(METRIC_STRATEGIES.keys())}")

print("\nMetric values:")
for name, strategy in METRIC_STRATEGIES.items():
    value = strategy.compute(y_test.flatten(), rf_pred.flatten())
    print(f"  {name} ({strategy.name}): {value:.4f}")

### 5.2 Per-Horizon Evaluation

In [ ]:
print("MAE per forecast horizon:")
for h in range(y_test.shape[1]):
    mae = mean_absolute_error(y_test[:, h], rf_pred[:, h])
    print(f"  Horizon {h + 1}: {mae:.4f}")

### 5.3 Multi-Model Comparison

In [ ]:
predictions_dict = {
    "RandomForest": rf_pred,
    "XGBoost": xgb_pred,
}

print("Model comparison:")
print("-" * 50)
print(f"{'Model':<15} {'MAE':>10} {'MSE':>12} {'RMSE':>10}")
print("-" * 50)

for name, pred in predictions_dict.items():
    mae = mean_absolute_error(y_test, pred)
    mse = mean_squared_error(y_test, pred)
    rmse = np.sqrt(mse)
    print(f"{name:<15} {mae:>10.4f} {mse:>12.4f} {rmse:>10.4f}")

---
## 6. DateTime Feature Engineering

### 6.1 engin_date() Method

In [ ]:
# Create fresh processor
processor = Processing()

# Apply datetime engineering
data_with_features = processor.engin_date(data, drop=False)

print(f"Original columns: {list(data.columns)}")
print(f"After engin_date: {list(data_with_features.columns)}")

In [ ]:
print("Generated datetime features:")
date_features = [c for c in data_with_features.columns if c.startswith("Date_")]
for feat in date_features:
    sample_val = data_with_features[feat].iloc[0]
    print(f"  {feat}: {sample_val}")

### 6.2 future_timestamps() Method

In [ ]:
# Generate future timestamps
last_date = data["Date"].iloc[-1]
future_dates = processor.future_timestamps(
    dataset=data,
    granularity="1mo",
    horizon=6,
)

print(f"Last date in data: {last_date}")
print(f"Future timestamps ({len(future_dates)}):")
for date in future_dates:
    print(f"  {date}")

---
## 7. Quick Reference

### Processing

```python
from tsforecasting import Processing

processor = Processing()

# Transform to supervised format
timeseries = processor.make_timeseries(
    dataset=data,
    window_size=10,
    horizon=5,
    datetime_engineering=True,
)

# DateTime features only
data_features = processor.engin_date(data, drop=False)

# Future timestamps
future = processor.future_timestamps(data, granularity="1mo", horizon=6)
```

### Direct Model Usage

```python
from tsforecasting.models import RandomForestForecaster, FORECASTER_CLASSES

# Option 1: Direct instantiation
model = RandomForestForecaster(n_estimators=100)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# Option 2: FORECASTER_CLASSES
model_class = FORECASTER_CLASSES["LightGBM"]
model = model_class(n_estimators=100)
```

### Expanding Window

```python
total = len(X)
initial_train = int(total * 0.8)
sliding_size = 10
iterations = (total - initial_train) // sliding_size

for i in range(iterations):
    train_end = initial_train + (i * sliding_size)
    X_train, X_test = X[:train_end], X[train_end:]
    y_train, y_test = y[:train_end], y[train_end:]

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
```

### Column Naming Convention

| Pattern | Description |
|---------|-------------|
| `y_lag_N` | Value at time t-N (input feature) |
| `y_horizon_N` | Value at time t+N (target variable) |
| `Date_*` | Datetime-derived features |